# 04 — Topic subgraphs: what was evaluated, how, and how to reproduce it

This notebook unpacks the 2026-07-13 topic-subgraphs evaluation
([`docs/eval/2026-07-13_topic_subgraphs.md`](../eval/2026-07-13_topic_subgraphs.md)) so you can
see **the exact questions asked, how they are asked, and where every reported number comes from**.

**The one-paragraph story.** Similarity top-k retrieval cannot answer *scoping* questions
("which of the sibling referral procedures charges fees regardless of initiator?") because the
answer needs *all* the sibling documents, and a top-k hit only lands on one. EMA's own topic hub
pages are curated indices, so we precompute each confirmed hub's qualified 2-hop `LINKS_TO`
fan-out as membership stamps (`:Document.topic_hubs`), and give the agent a `topic_context` tool
that returns the **complete member catalog** of a topic. Recipe `topic_agent` = the existing
`steered_agent` + this tool. On the 10 scoping (T2) benchmark questions it scored
**5.000/5.000** (correctness/faithfulness) vs the baseline's **4.700/4.900** — and the two
baseline failures were exactly the cross-sibling misses the mechanism targets.

**Sections** — 1: the benchmark questions · 2: the subgraph under the hood (Neo4j, no LLM) ·
3: how one question is asked end-to-end (needs `ANTHROPIC_API_KEY`, off by default) ·
4: the eval mechanics (what `scripts/run_eval.py` does) · 5: reproduce the reported tables
from the MLflow store (offline).

**Needs:** Mongo + Neo4j up (`scripts/start_services.sh`) and a built graph with stamped
memberships for §2; only the local `mlflow.db` for §5.

In [1]:
# Environment: credentials live in ~/.myenvs/ema_nlp.env (never in the repo).
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.home() / ".myenvs" / "ema_nlp.env")
os.environ.setdefault("NEO4J_URI", "bolt://localhost:7688")  # project container (alt port)

# Make `harness` importable when the notebook is started from docs/examples/.
REPO = Path.cwd()
while not (REPO / "harness").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.chdir(REPO)

RUN_LLM = False  # flip to True to run §3's live agent turn (costs API tokens)
print("repo root:", REPO)

repo root: /home/moritz/github_repos/ema_nlp


## 1. The questions — `benchmark/benchmark.jsonl`

45 hand-curated items with **gold answers**, in four types (always reported per type,
never aggregate — the CLAUDE.md eval rule):

| Type | n | What it tests |
|---|---|---|
| T1 Lookup | 20 | one fact from one document |
| T2 Scoping | 10 | *comparing sibling procedures* — the subgraph target |
| T3 Multi-hop | 10 | chaining facts across documents |
| T4 Synthesis | 5 | multi-document comparative write-ups |

Every T2 item draws on the **referral-procedures Q&A family** (three sibling
`questions-answers-article-3x` documents) — cross-sibling by construction. That is why the
`referral_procedures` hub was built first, and also the honest caveat: the T2 win proves the
mechanism on one topic family, not breadth.

In [2]:
import json

import pandas as pd

rows = [json.loads(l) for l in open("benchmark/benchmark.jsonl", encoding="utf-8")]
df = pd.DataFrame(rows)
print(df["type"].value_counts().sort_index().to_string())

t2 = df[df["type"] == "T2"][["bench_id", "question", "gold_answer"]].copy()
t2["gold_answer"] = t2["gold_answer"].str.slice(0, 140) + "…"
pd.set_option("display.max_colwidth", 120)
t2  # the ten scoping questions the head-to-head was scored on

type
T1    20
T2    10
T3    10
T4     5


,bench_id,question,gold_answer
20,T2-001,Which EMA referral procedure uses the PRAC as the lead scientific assessment committee?,The Article 31 pharmacovigilance referral uses the PRAC as the lead scientific assessment committee. The PRAC appoin...
21,T2-002,In which EMA referral procedure are fees always levied by the Agency regardless of who initiated the procedure?,"In Article 31 pharmacovigilance referrals, the Agency always levies a fee. The share payable by each MAH is calculat..."
22,T2-003,Who is the default contact person for an MAH receiving EMA correspondence in an Article 31 pharmacovigilance referral?,The Qualified Person for Pharmacovigilance (QPPV) is by default the contact person for an MAH during an Article 31 p...
23,T2-004,Which specific EMA referral procedure applies when Member States have adopted divergent authorisation decisions for ...,The Article 30 'harmonisation' referral procedure applies when divergent decisions have been adopted by Member State...
24,T2-005,What is the standard initial active-day assessment period that distinguishes Article 31 pharmacovigilance referrals ...,Article 31 pharmacovigilance referrals follow a standard initial PRAC assessment period of 30 active days (extendabl...
25,T2-006,An EMA referral is being initiated following new post-marketing safety signals from pharmacovigilance data for an au...,An Article 31 pharmacovigilance referral should be initiated where the interests of the Union are involved as a resu...
26,T2-007,"When a centrally authorised product (CAP) is included in an Article 31 pharmacovigilance referral, who at the MAH is...","When a centrally authorised product is involved in an Article 31 pharmacovigilance referral, the Qualified Person fo..."
27,T2-008,For which referral procedure is the scope of included medicinal products limited exclusively to the product(s) for w...,"Under Article 30 referral procedures, only the concerned medicinal product for which divergent decisions have been a..."
28,T2-009,Can MAHs from different company groups pool their responses and present a single consolidated submission in an Artic...,Yes. MAHs/applicants in an Article 31 non-pharmacovigilance referral can form a group for the purpose of the procedu...
29,T2-010,Under which EMA referral procedure are fees payable only when the MAH or applicant was the one who initiated the ref...,Fees are conditional on MAH/applicant initiation in both Article 30 referrals and Article 31 non-pharmacovigilance r...


## 2. The subgraph under the hood (Neo4j only — no LLM)

The hub seed list is pure data (`harness/configs/hubs/default.yaml`): one confirmed hub,
`referral_procedures`, with per-hub walk bounds (2 hops; every node on the path must match
`category IN [...] OR doc_type IN [...]`; Veterinary/Corporate audiences excluded).
`scripts/manage_topic_hubs.py build` ran this walk offline and stamped the 49 member URLs
into Mongo `document_metadata` → propagated to `:Document.topic_hubs`.

The cell below re-runs the walk read-only and re-checks the **step-5 gate**: the 3 gold
documents behind all 10 T2 items must be members.

In [3]:
from harness.indexing.property_graph import neo4j_store_from_env
from harness.indexing.subgraphs import composition_histogram, walk_members
from harness.retrieval.hubs import load_hubs

store = neo4j_store_from_env()
hub = load_hubs().get("referral_procedures")
members = walk_members(store, hub)
print(f"{len(members)} members; by category:",
      dict(composition_histogram(members, "category")))

gold_markers = ["questions-answers-article-31-pharmacovigilance",
                "questions-answers-article-30",
                "questions-answers-article-31-non-pharmacovigilance"]
urls = {m["url"] for m in members}
for marker in gold_markers:
    ok = any(marker in u for u in urls)
    print(("PASS " if ok else "FAIL "), marker)

49 members; by category: {'qa': 28, 'regulatory_overview': 14, 'scientific_guideline': 6, 'regulatory_procedure': 1}
PASS  questions-answers-article-31-pharmacovigilance
PASS  questions-answers-article-30
PASS  questions-answers-article-31-non-pharmacovigilance


What the agent actually receives from the tool: a pageable, query-ranked catalog with an
honest header (total, page x/y, `truncated` flag), PDF revisions grouped under their HTML
detail page. This is `topic_context` exactly as the agent calls it (needs the embed model,
~1.3 GB on first download; fine on CPU):

In [4]:
from harness.indexing import build_retriever, load_index_profile, open_index
from harness.retrieval.subgraphs import SubgraphPolicy
from harness.tools import get_tool

profile = load_index_profile("neo4j_steered")
retriever = build_retriever(profile, open_index(profile))
tool = get_tool("topic_context", retriever=retriever, hubs=load_hubs(),
                subgraph=SubgraphPolicy(context="map", page_size=10))
out = tool.fn(topic="referral_procedures",
              query="which referral procedure charges fees regardless of who initiated it?")
print(out[:1800])

/home/moritz/github_repos/ema_nlp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6953.07it/s]

LLM is explicitly disabled. Using MockLLM.
LLM is explicitly disabled. Using MockLLM.


[topic: referral_procedures — 49 documents in 25 groups; page 1/3; truncated=true — call topic_context(topic='referral_procedures', page=2) for more]
1. Questions and answers: Article 30 referral procedures [qa] — https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation/referral-procedures-human-medicines/questions-answers-article-30-referral-procedures
   - questions and answers article 30 referral procedures en [regulatory-procedural-guideline] — https://www.ema.europa.eu/en/documents/regulatory-procedural-guideline/questions-and-answers-article-30-referral-procedures_en.pdf
2. Questions and answers: Article 13 referral procedures [qa] — https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation/referral-procedures-human-medicines/questions-answers-article-13-referral-procedures
   - questions and answers article 13 referral procedures en [regulatory-procedural-guideline] — https://www.ema.europa.eu/en/documents/regulatory-procedural-guideline/question

## 3. How one question is asked, end-to-end (optional — costs API tokens)

The eval asks questions **exactly like the chat UI does**: `build_recipe("topic_agent")`
assembles a LlamaIndex `FunctionAgent` with the recipe's system prompt
(`harness/prompts/agent_topic.md`), three tools (`ema_search`, `resolve_substance`,
`topic_context`) and the `RegulatoryAnswer` structured-output schema; the benchmark
`question` string is passed as the user message, nothing else. The agent decides itself
whether/when to call `topic_context` — the prompt only *suggests* it for scoping/enumeration
questions. Flip `RUN_LLM = True` in the setup cell to watch one T2 question run.

In [5]:
if RUN_LLM:
    from harness.recipes import build_recipe, get_recipe
    from harness.tools.search import capture_search_nodes

    runner = build_recipe(get_recipe("topic_agent"), open_index(profile))
    q = t2.iloc[2]["question"]  # a fee question — one the baseline got wrong
    print("Q:", q)
    with capture_search_nodes() as nodes:
        result = runner.invoke({"user_msg": q})
    print("\nA:", result["answer_text"][:900])
    origins = [n.node.metadata.get("retrieval_origin") for n in nodes]
    print("\nevidence nodes by origin:", {o: origins.count(o) for o in set(origins)})
else:
    print("RUN_LLM is False — skipping the live agent turn (see §5 for recorded runs)")

RUN_LLM is False — skipping the live agent turn (see §5 for recorded runs)


## 4. The eval mechanics — what `scripts/run_eval.py` does

```bash
python scripts/run_eval.py --recipe topic_agent   --types T2                        # Opus (recipe default)
python scripts/run_eval.py --recipe steered_agent --types T2
python scripts/run_eval.py --recipe topic_agent   --types T1 T3 T4 --model claude_sonnet
```

Per question type it: (1) wraps the recipe as an `mlflow.genai` `predict_fn` (which also
captures the retrieved passages), (2) asks every question of that type, (3) scores each
answer with **two LLM judges on Claude Opus** (1–5): `correctness` against the item's
`gold_answer`, and `faithfulness` against the passages the agent actually retrieved,
(4) logs **one MLflow run per type**, tagged `ema.recipe` / `ema.question_type`, with every
question's full trace (tool calls included) attached. MLflow is the system of record —
the report tables are *views of these runs*, which §5 reproduces.

Model note: the T2 head-to-head generated with **Opus** on both recipes (fair comparison);
the later cross-type sweep generated with **Sonnet 5** (budget) — judges stayed Opus
throughout. Never compare numbers across the two generation models.

## 5. Reproduce the reported tables from `mlflow.db` (offline)

Reads the local MLflow store directly: every eval run for the two recipes, its per-type
means, and the per-item scores + tool usage behind them.

In [6]:
import datetime

import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = mlflow.MlflowClient()

runs = []
for exp in client.search_experiments():
    for r in client.search_runs([exp.experiment_id], max_results=100,
                                order_by=["attributes.start_time ASC"]):
        t = r.data.tags
        if t.get("ema.recipe") in ("topic_agent", "steered_agent") and r.data.metrics:
            runs.append({
                "run_id": r.info.run_id[:8],
                "start": datetime.datetime.fromtimestamp(r.info.start_time/1000)
                          .strftime("%m-%d %H:%M"),
                "recipe": t["ema.recipe"],
                "type": t.get("ema.question_type"),
                **{k: round(v, 3) for k, v in r.data.metrics.items()},
                "_id": r.info.run_id, "_exp": r.info.experiment_id,
            })
summary = pd.DataFrame(runs)
summary.drop(columns=["_id", "_exp"])
# The report uses: 30ceab33/218c45c3 (T2 head-to-head, Opus), 1239ec66 (T1), 0a22ab4b (T3),
# 41c3cb35 (T4) — all topic_agent on Sonnet 5. 5ba51e5c is the PARTIAL Opus T1 run from the
# credit-exhausted sweep (17/20 judged); c57fb04a is a 1-question smoke test. Exclude those two.

,run_id,start,recipe,type,correctness_mean,faithfulness_mean
0,30ceab33,07-13 13:40,topic_agent,T2,5.000,5.000
1,218c45c3,07-13 13:42,steered_agent,T2,4.700,4.900
2,5ba51e5c,07-13 13:46,topic_agent,T1,4.353,4.765
3,c57fb04a,07-13 14:20,topic_agent,T1,3.000,4.000
4,1239ec66,07-13 14:22,topic_agent,T1,4.263,4.526
5,0a22ab4b,07-13 14:25,topic_agent,T3,4.600,4.600
6,41c3cb35,07-13 15:46,topic_agent,T4,3.800,5.000


In [7]:
def per_item(run_row):
    """Per-question judge scores + tool usage for one eval run (from its traces)."""
    out = []
    for tr in client.search_traces(locations=[run_row["_exp"]], run_id=run_row["_id"],
                                   max_results=25):
        full = client.get_trace(tr.info.trace_id)
        scores = {a.name: a.value for a in (full.info.assessments or [])
                  if a.name in ("correctness", "faithfulness")}
        tools = []
        for span in full.data.spans:
            if span.span_type == "TOOL":
                blob = json.dumps({k: str(v)[:300] for k, v in (span.attributes or {}).items()})
                for cand in ("topic_context", "ema_search", "resolve_substance"):
                    if cand in blob:
                        tools.append(cand)
                        break
        req = full.data.request
        q = json.loads(req) if isinstance(req, str) else (req or {})
        out.append({"question": (q.get("user_msg") or "")[:75],
                    "correctness": scores.get("correctness"),
                    "faithfulness": scores.get("faithfulness"),
                    "topic_context": tools.count("topic_context"),
                    "ema_search": tools.count("ema_search")})
    return pd.DataFrame(out)

head_to_head = summary[summary["type"] == "T2"]
for _, row in head_to_head.iterrows():
    print(f"\n=== {row['recipe']} T2 (run {row['run_id']}) ===")
    display(per_item(row))
# Look for: steered_agent's correctness 3 + 4 items are the two FEE questions;
# topic_agent used topic_context (column = 1) on both and scored 5.


=== topic_agent T2 (run 30ceab33) ===


,question,correctness,faithfulness,topic_context,ema_search
0,When a centrally authorised product (CAP) is included in an Article 31 phar,5,5,0,2
1,An EMA referral is being initiated following new post-marketing safety sign,5,5,1,1
2,Under which EMA referral procedure are fees payable only when the MAH or ap,5,5,1,1
3,For which referral procedure is the scope of included medicinal products li,5,5,1,3
4,Can MAHs from different company groups pool their responses and present a s,5,5,0,5
5,What is the standard initial active-day assessment period that distinguishe,5,5,0,2
6,Which specific EMA referral procedure applies when Member States have adopt,5,5,1,1
7,In which EMA referral procedure are fees always levied by the Agency regard,5,5,1,6
8,Who is the default contact person for an MAH receiving EMA correspondence i,5,5,0,3
9,Which EMA referral procedure uses the PRAC as the lead scientific assessmen,5,5,1,1



=== steered_agent T2 (run 218c45c3) ===


,question,correctness,faithfulness,topic_context,ema_search
0,For which referral procedure is the scope of included medicinal products li,5,5,0,4
1,Under which EMA referral procedure are fees payable only when the MAH or ap,3,5,0,7
2,Can MAHs from different company groups pool their responses and present a s,5,5,0,6
3,An EMA referral is being initiated following new post-marketing safety sign,5,5,0,2
4,When a centrally authorised product (CAP) is included in an Article 31 phar,5,5,0,1
5,What is the standard initial active-day assessment period that distinguishe,5,5,0,2
6,Which specific EMA referral procedure applies when Member States have adopt,5,5,0,2
7,Who is the default contact person for an MAH receiving EMA correspondence i,5,5,0,2
8,In which EMA referral procedure are fees always levied by the Agency regard,4,4,0,6
9,Which EMA referral procedure uses the PRAC as the lead scientific assessmen,5,5,0,4


In [8]:
# The cross-type sweep rows (topic_agent, Sonnet 5). Same drill-down works for any run:
sweep = summary[(summary["recipe"] == "topic_agent") & (summary["type"] != "T2")
                & (~summary["run_id"].isin(["5ba51e5c", "c57fb04a"]))]
for _, row in sweep.iterrows():
    print(f"\n=== topic_agent {row['type']} (run {row['run_id']}) ===")
    items = per_item(row)
    display(items.sort_values("correctness", na_position="first").head(6))
# Note the pattern in the low scorers: worksharing questions — a topic with NO built
# subgraph yet. That observation drives docs/next/topic_subgraphs_followups.md item 2.


=== topic_agent T1 (run 1239ec66) ===


,question,correctness,faithfulness,topic_context,ema_search
13,How far in advance must an MAH submit a letter of intent before applying fo,None,.The MAH is expected to submit a letter of intent to the EMA at least 3 months before their intention to submit the ...,0,29
12,Is there a fee payable for Type IB variation applications under a workshari,2,3,0,5
10,Is the CHMP co-rapporteur normally involved in the assessment of an extensi,2,2,0,7
19,How far in advance must an MAH inform the Agency before submitting a variat,3,3,0,2
1,What is the standard initial PRAC recommendation period for an Article 31 p,3,5,1,1
14,How far in advance of submission must an MAH notify EMA of their intention,3,3,0,16



=== topic_agent T3 (run 0a22ab4b) ===


,question,correctness,faithfulness,topic_context,ema_search
5,For a worksharing application that includes centrally authorised products (,3,5,0,8
4,Which medicinal products are included in an Article 31 pharmacovigilance re,4,5,1,1
6,"In an Article 31 pharmacovigilance referral, what happens after the PRAC is",4,5,1,2
0,"For traditional herbal medicinal product registration, can medicinal use do",5,5,0,4
1,When a centrally authorised product is involved in an Article 31 pharmacovi,5,5,1,1
2,"If an MAH receives an unfavourable CHMP opinion in a worksharing procedure,",5,5,0,4



=== topic_agent T4 (run 41c3cb35) ===


,question,correctness,faithfulness,topic_context,ema_search
4,What are the key differences in fee arrangements between Article 30 referra,1,None,1,19
0,An MAH holds a herbal medicinal product that has 35 years of documented med,4,5,0,8
2,What are the key differences in how the assessment is conducted and the fin,4,5,0,14
1,How do the pre-submission advance notice requirements compare across three,5,agent errored,0,21
3,How does the scope of medicinal products included in an Article 30 referral,5,5,0,11


## 6. Where things stand

- **Established:** the precompute→lookup mechanism works live end-to-end; the agent adopts
  `topic_context` unprompted (6/10 T2, 9/20 T1); on T2 it removes exactly the baseline's
  cross-sibling completeness failures at the same model and judges.
- **Open** (all in [`docs/next/topic_subgraphs_followups.md`](../next/topic_subgraphs_followups.md)):
  the `steered_agent` T1/T3/T4 baseline for the formal no-regression verdict; more hubs
  (worksharing first — the sweep's failures concentrate there); 2–3 cross-family T2 items;
  a client-side LLM request timeout (one eval leg hung silently for ~70 min).
- **Full detail:** [`docs/eval/2026-07-13_topic_subgraphs.md`](../eval/2026-07-13_topic_subgraphs.md);
  design + design history: [`docs/next/topic_subgraphs.md`](../next/topic_subgraphs.md);
  shipped surface: [`docs/RETRIEVAL.md`](../RETRIEVAL.md) §7.1.